# Weather cohort exploration

Group construction events into weather cohorts and preview a self-driving impact score.
Exploration only: the production logic lives in the dbt models. Every row here comes from
`int_construction_weather_features`, so null handling and features are identical to production.

Kernel: **Python (weather-cohort)** (repo-root `.venv`). Severity is intentionally not used: it describes
the construction, not the weather.

## 1. Setup and connection test

Connection settings come from `notebooks/.env` (see `.env.example`) or, per setting, the dbt profile
`road_construction_autonomy` in `~/.dbt/profiles.yml`. Auth uses gcloud Application Default Credentials
unless `GOOGLE_APPLICATION_CREDENTIALS` is set. No credentials are stored in this notebook.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from dotenv import load_dotenv
from google.cloud import bigquery

# Notebook may be launched from the repo root or from notebooks/
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

load_dotenv(NOTEBOOK_DIR / ".env")

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

In [ ]:
def load_dbt_target(profile="road_construction_autonomy"):
    """Return the active target from the dbt profile, or {} if unavailable."""
    profiles_dir = Path(os.environ.get("DBT_PROFILES_DIR", Path.home() / ".dbt"))
    path = profiles_dir / "profiles.yml"
    if not path.exists():
        return {}
    prof = (yaml.safe_load(path.read_text()) or {}).get(profile, {})
    return prof.get("outputs", {}).get(prof.get("target"), {})


dbt_target = load_dbt_target()

BQ_PROJECT = os.getenv("BQ_PROJECT") or dbt_target.get("project")
BQ_LOCATION = os.getenv("BQ_LOCATION") or dbt_target.get("location", "US")
# dbt's default schema naming: <target dataset>_<custom schema> (see +schema in dbt_project.yml)
BQ_DATASET = os.getenv("BQ_DATASET") or (
    f"{dbt_target['dataset']}_dwh_road_construction" if dbt_target.get("dataset") else None
)
BQ_INT_DATASET = os.getenv("BQ_INT_DATASET") or (
    f"{dbt_target['dataset']}_int_road_construction" if dbt_target.get("dataset") else None
)

assert BQ_PROJECT and BQ_DATASET and BQ_INT_DATASET, (
    "Set BQ_PROJECT / BQ_DATASET / BQ_INT_DATASET in notebooks/.env or configure the dbt profile."
)

FACT_TABLE = f"`{BQ_PROJECT}.{BQ_DATASET}.fact_construction`"
FEATURES_TABLE = f"`{BQ_PROJECT}.{BQ_INT_DATASET}.int_construction_weather_features`"
client = bigquery.Client(project=BQ_PROJECT, location=BQ_LOCATION)

print(f"project:  {BQ_PROJECT}")
print(f"location: {BQ_LOCATION}")
print(f"fact:     {FACT_TABLE}")
print(f"features: {FEATURES_TABLE}")

In [ ]:
# Connection test
row_count = client.query(f"SELECT COUNT(*) AS n FROM {FACT_TABLE}").to_dataframe()["n"].iloc[0]
print(f"fact_construction rows: {row_count:,}")

## 2. Stratified sample

The sample is built in dbt by `int_weather_cohort_training_sample`, the same table BigQuery ML trains on,
so the notebook and production use identical rows. Strata are month-of-year x `precip_type`
(`unknown` excluded); each stratum gets its proportional share of `weather_cohort_sample_target_rows`
(400k) but at least `weather_cohort_sample_min_per_stratum` (2,000) rows, or all of them if smaller, so rare
conditions such as ice and heavy snow are kept. Selection is deterministic
(`FARM_FINGERPRINT(construction_id:random_state)`). `sample_weight = stratum_rows / stratum_quota` scales
each row back to the population for weighted profiles.

The pull is cached to `notebooks/data/sample.parquet` (gitignored). Delete the file, or set
`REFRESH_SAMPLE = True`, after the dbt models change.

In [ ]:
REFRESH_SAMPLE = False
SAMPLE_PATH = DATA_DIR / "sample.parquet"
SAMPLE_TABLE = f"`{BQ_PROJECT}.{BQ_INT_DATASET}.int_weather_cohort_training_sample`"

if SAMPLE_PATH.exists() and not REFRESH_SAMPLE:
    sample = pd.read_parquet(SAMPLE_PATH)
    print(f"loaded cached sample: {len(sample):,} rows from {SAMPLE_PATH}")
else:
    t0 = time.time()
    job = client.query(f"SELECT * EXCEPT (is_init_centroid) FROM {SAMPLE_TABLE} ORDER BY construction_id")
    sample = job.to_dataframe()
    sample = sample.drop(columns=[c for c in sample.columns if c.startswith("feat_")])
    sample.to_parquet(SAMPLE_PATH, index=False)
    print(f"pulled {len(sample):,} rows in {time.time() - t0:.1f}s "
          f"({job.total_bytes_processed / 2**20:.0f} MiB processed, {job.slot_millis:,} slot-ms)")

(sample.groupby("precip_type")
    .agg(sample_rows=("construction_id", "size"), population_rows=("sample_weight", "sum"))
    .assign(population_rows=lambda d: d.population_rows.round().astype(int))
    .sort_values("sample_rows", ascending=False))

## 3. Feature engineering

Inputs come straight from `int_construction_weather_features`; imputation, invalid-value handling,
clipping, precip_type, intensity, wet_road, freezing_wet and is_night are all computed in dbt. This section
only encodes and scales them:

- numeric: `precipitation_inches_capped`, `visibility_miles_clipped` (0-10 mi), `wind_speed_mph_clipped`
  (0-50 mph), `humidity_percent`, `temperature_fahrenheit_clipped` (-30F to 120F)
- `precip_type`: one-hot over the 7 known types (an `unknown` row is all zeros, so unknown rows can be
  assigned to the nearest cohort later)
- `precip_intensity`: ordinal none 0, light 1, moderate 2, heavy 3
- flags: `wet_road`, `freezing_wet`, `is_blowing`, `is_night`
- all columns standardized (z-scores), as BigQuery ML does with `standardize_features = true`

Three row-level markers are used only to evaluate cohorts (not as features): heavy precipitation
(`precip_intensity = 'heavy'`), dense fog (fog with visibility < 1 mi) and high wind (>= 25 mph).

In [ ]:
from sklearn.preprocessing import StandardScaler

PRECIP_TYPES = ["clear_cloudy", "rain", "snow", "ice_freezing", "fog", "haze_smoke_dust", "thunderstorm"]
TYPE_COLUMNS = [f"type_{t}" for t in PRECIP_TYPES]
INTENSITY_ORDINAL = {"none": 0, "light": 1, "moderate": 2, "heavy": 3}
NUMERIC_FEATURES = ["precipitation_inches_capped", "visibility_miles_clipped", "wind_speed_mph_clipped",
                    "humidity_percent", "temperature_fahrenheit_clipped"]
FLAG_FEATURES = ["wet_road", "freezing_wet", "is_blowing", "is_night"]


def build_features(d: pd.DataFrame) -> pd.DataFrame:
    """Encode int_construction_weather_features rows into the clustering matrix (unscaled)."""
    X = d[NUMERIC_FEATURES].astype(float).copy()
    for t in PRECIP_TYPES:
        X[f"type_{t}"] = (d["precip_type"] == t).astype(float)
    X["precip_intensity_ord"] = d["precip_intensity"].map(INTENSITY_ORDINAL).astype(float)
    for col in FLAG_FEATURES:
        X[col] = d[col].astype(float)
    return X


X = build_features(sample)
scaler = StandardScaler().fit(X)
Z_base = scaler.transform(X)
assert not np.isnan(Z_base).any(), "unexpected nulls in features"

markers = pd.DataFrame({
    "heavy_precip": sample["precip_intensity"].eq("heavy"),
    "dense_fog": sample["precip_type"].eq("fog") & (sample["visibility_miles_clipped"] < 1),
    "high_wind": sample["wind_speed_mph_clipped"] >= 25,
}, index=sample.index)

print(f"feature matrix: {Z_base.shape[0]:,} rows x {Z_base.shape[1]} columns")
print("marker rows in sample:", markers.sum().to_dict())
X.describe().T[["mean", "std", "min", "max"]].round(2)

## 4. Choosing k

### 4a. MiniBatchKMeans sweep (all columns standardized)

MiniBatchKMeans for k = 4-10; silhouette on a fixed 50k-row subsample.

In [ ]:
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

SILHOUETTE_ROWS = 50_000
KMEANS_PARAMS = dict(batch_size=4096, n_init=10, max_iter=300)
sil_idx = np.random.default_rng(RANDOM_STATE).choice(len(Z_base), SILHOUETTE_ROWS, replace=False)
is_clear = (sample["precip_type"] == "clear_cloudy").values

mbk_sweep = []
for k in range(4, 11):
    m = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, **KMEANS_PARAMS).fit(Z_base)
    mbk_sweep.append({"k": k, "silhouette": silhouette_score(Z_base[sil_idx], m.labels_[sil_idx], random_state=RANDOM_STATE)})
mbk_sweep = pd.DataFrame(mbk_sweep).set_index("k")

# stability of MiniBatchKMeans vs full KMeans at k = 9 across three seeds
stab = []
for name, make in [("MiniBatchKMeans", lambda s: MiniBatchKMeans(n_clusters=9, random_state=s, **KMEANS_PARAMS)),
                   ("KMeans", lambda s: KMeans(n_clusters=9, random_state=s, n_init=10))]:
    labs = {s: make(s).fit(Z_base).labels_ for s in (RANDOM_STATE, 7, 123)}
    for s in (7, 123):
        a, b = labs[RANDOM_STATE], labs[s]
        stab.append({"model": name, "vs_seed": s, "ari_all": adjusted_rand_score(a, b),
                     "ari_precip_rows": adjusted_rand_score(a[~is_clear], b[~is_clear]),
                     "ari_clear_rows": adjusted_rand_score(a[is_clear], b[is_clear])})
display(pd.DataFrame(stab).round(3))
mbk_sweep.round(3)

MiniBatchKMeans keeps the precipitation clusters stable across seeds but reshuffles the clear-weather
split; full KMeans is identical across seeds and still fast at this size, so **every model from here on
uses full KMeans**.

### 4b. Down-weighting the categorical columns

With every column standardized, the 7 one-hot `precip_type` columns dominate the distances and the
clusters mostly reproduce `precip_type`. Three weightings are compared for k = 6-12:

- **baseline**: all standardized columns weight 1 (the earlier k = 9 proposal)
- **type_as_one**: the one-hot block is scaled by 1/sqrt(7), so its total variance equals one standardized
  feature (the categorical contributes like a single feature); intensity is already a single column
- **type_half**: the one-hot block at half that weight, and `precip_intensity_ord` at 0.5

For each: silhouette (in that weighting's space, so compare within a weighting rather than across),
stability (minimum ARI vs seeds 7 and 123), naming (clusters that get no rule-based name, or share a
name with another cluster), and whether heavy precipitation, dense fog and high wind get their own
cohort (a cluster where >= 50% of rows have the marker and that holds >= 50% of all marker rows).
Results are cached to `notebooks/data/k_grid.pkl`.

In [ ]:
import pickle

WEIGHT_SCHEMES = {
    "baseline": {},
    "type_as_one": {c: 1 / np.sqrt(len(TYPE_COLUMNS)) for c in TYPE_COLUMNS},
    "type_half": {**{c: 0.5 / np.sqrt(len(TYPE_COLUMNS)) for c in TYPE_COLUMNS}, "precip_intensity_ord": 0.5},
}


def scheme_weights(scheme):
    return np.array([WEIGHT_SCHEMES[scheme].get(c, 1.0) for c in X.columns])


# --- rule-based cohort names from a cluster's weighted average profile (stable across refits)
LABEL_SHARE = 0.5


def label_cluster(p: pd.Series) -> str:
    if p["type_ice_freezing"] >= LABEL_SHARE:
        return "Freezing/Icy"
    if p["type_snow"] >= LABEL_SHARE:
        return "Heavy Snow" if p["heavy_precip"] >= LABEL_SHARE else "Snow"
    if p["type_thunderstorm"] >= LABEL_SHARE:
        return "Heavy Thunderstorm" if p["heavy_precip"] >= LABEL_SHARE else "Thunderstorm"
    if p["type_rain"] >= LABEL_SHARE:
        return "Heavy Rain" if p["heavy_precip"] >= LABEL_SHARE else "Rain"
    if p["type_fog"] >= LABEL_SHARE:
        return "Dense Fog" if p["dense_fog"] >= LABEL_SHARE else "Fog/Low Visibility"
    if p["is_blowing"] >= LABEL_SHARE or p["high_wind"] >= LABEL_SHARE:
        return "High Wind/Blowing"
    if p["type_haze_smoke_dust"] >= LABEL_SHARE:
        return "Haze/Smoke/Dust"
    if p["type_clear_cloudy"] >= LABEL_SHARE:
        return "Clear/Dry Night" if p["is_night"] >= LABEL_SHARE else "Clear/Dry Day"
    if p["heavy_precip"] >= LABEL_SHARE:
        return "Heavy Precipitation"
    if p["freezing_wet"] >= LABEL_SHARE:
        return "Freezing/Icy"
    return "Mixed"


def cohort_profile(lab):
    """Population-weighted profile per cluster (original units) plus marker shares and a label."""
    frame = X.join(markers.astype(float))
    w = sample["sample_weight"].to_numpy()
    sums = frame.mul(w, axis=0).groupby(lab).sum()
    wsum = pd.Series(w).groupby(lab).sum()
    prof = sums.div(wsum.values, axis=0)
    prof["sample_rows"] = pd.Series(lab).value_counts().sort_index().values
    prof["population_share"] = (wsum / wsum.sum()).values
    names = prof.apply(label_cluster, axis=1)
    # duplicates get a numeric suffix, ordered by population share
    order = prof["population_share"].sort_values(ascending=False).index
    rank = names.loc[order].groupby(names.loc[order]).cumcount()
    names.loc[order] = [n if r == 0 else f"{n} ({r + 1})" for n, r in zip(names.loc[order], rank)]
    prof["cohort_label"] = names
    prof["duplicate_name"] = rank.reindex(prof.index) > 0
    return prof


def marker_capture(lab, marker):
    """Best cluster for a marker: purity (share of the cluster with the marker) and coverage."""
    m = markers[marker].to_numpy()
    ct = pd.crosstab(lab, m)
    if True not in ct.columns:
        return 0.0, 0.0
    purity = ct[True] / ct.sum(axis=1)
    coverage = ct[True] / ct[True].sum()
    best = np.minimum(purity, coverage).idxmax()
    return float(purity[best]), float(coverage[best])


GRID_K = range(6, 13)
SEEDS = (RANDOM_STATE, 7, 123)
GRID_PATH = DATA_DIR / "k_grid.pkl"
grid_key = (tuple(sample["construction_id"].iloc[[0, -1]]), len(sample), tuple(WEIGHT_SCHEMES), tuple(GRID_K), SEEDS)

if GRID_PATH.exists() and pickle.loads(GRID_PATH.read_bytes())["key"] == grid_key:
    grid_cache = pickle.loads(GRID_PATH.read_bytes())
    print(f"loaded cached grid from {GRID_PATH}")
else:
    grid_cache = {"key": grid_key, "rows": [], "labels": {}}
    t_start = time.time()
    for scheme in WEIGHT_SCHEMES:
        Zw = Z_base * scheme_weights(scheme)
        for k in GRID_K:
            labs = {s: KMeans(n_clusters=k, random_state=s, n_init=10).fit(Zw).labels_ for s in SEEDS}
            lab = labs[RANDOM_STATE]
            prof = cohort_profile(lab)
            row = {"scheme": scheme, "k": k,
                   "silhouette": silhouette_score(Zw[sil_idx], lab[sil_idx], random_state=RANDOM_STATE),
                   "ari_min_vs_seeds": min(adjusted_rand_score(lab, labs[s]) for s in SEEDS[1:]),
                   "unnamed": int((prof["cohort_label"] == "Mixed").sum()),
                   "duplicate_names": int(prof["duplicate_name"].sum())}
            for mk in markers.columns:
                pur, cov = marker_capture(lab, mk)
                row[f"{mk}_purity"], row[f"{mk}_coverage"] = round(pur, 2), round(cov, 2)
                row[f"{mk}_own"] = pur >= 0.5 and cov >= 0.5
            row["labels"] = ", ".join(prof.sort_values("population_share", ascending=False)["cohort_label"])
            grid_cache["rows"].append(row)
            grid_cache["labels"][(scheme, k)] = lab
            print(f"{scheme:12s} k={k:2d} done ({time.time() - t_start:.0f}s)")
    GRID_PATH.write_bytes(pickle.dumps(grid_cache))

# naming depends on the label rules, so recompute it from the cached cluster labels
for row in grid_cache["rows"]:
    prof = cohort_profile(grid_cache["labels"][(row["scheme"], row["k"])])
    row["unnamed"] = int((prof["cohort_label"] == "Mixed").sum())
    row["duplicate_names"] = int(prof["duplicate_name"].sum())
    row["labels"] = ", ".join(prof.sort_values("population_share", ascending=False)["cohort_label"])

grid = pd.DataFrame(grid_cache["rows"])
grid[["scheme", "k", "silhouette", "ari_min_vs_seeds", "unnamed", "duplicate_names",
      "heavy_precip_own", "dense_fog_own", "high_wind_own"]].round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for scheme, g in grid.groupby("scheme"):
    ax[0].plot(g["k"], g["silhouette"], marker="o", label=scheme)
    ax[1].plot(g["k"], g["ari_min_vs_seeds"], marker="o", label=scheme)
ax[0].set_title("Silhouette vs k (each in its own weighted space)"); ax[1].set_title("Stability: min ARI vs 2 other seeds")
for a in ax:
    a.set_xlabel("k"); a.legend()
plt.tight_layout(); plt.show()

pd.set_option("display.max_colwidth", 400)
grid[["scheme", "k", "heavy_precip_purity", "heavy_precip_coverage", "dense_fog_purity", "dense_fog_coverage",
      "high_wind_purity", "high_wind_coverage", "labels"]]

### 4c. Choice

**Recommendation: keep k = 9 with all columns standardized (the baseline proposal).** Down-weighting the
categorical columns did not produce clearly better, nameable, stable cohorts:

- With the type block weakened (`type_as_one`, `type_half`), K-Means spends its extra clusters splitting clear
  weather by temperature and humidity: almost every k has a "Mixed" cluster and/or several duplicate
  "Clear/Dry Day" clusters, and several are unstable across seeds (ARI 0.58-0.69).
- **Heavy precipitation** gets its own cohort only with `type_as_one` at k = 10-11 (purity 0.62, coverage
  0.76), and those solutions also contain a "Mixed" cluster and two duplicate names. Baseline k = 10-12 adds
  a "Heavy Precipitation" cohort too, but it holds only a third of heavy-precipitation rows (and k = 10 is
  unstable).
- **Dense fog** never gets its own cohort: fog stays a single cohort under every weighting (47% of its rows
  are below 1 mi).
- **High wind** never gets a cohort with purity above 0.5.
- Baseline k = 9 is perfectly stable (ARI 1.000 across seeds) with no unnamed or duplicate clusters.

The within-type risk tail (heavy precipitation, dense fog, high wind) is therefore carried by the impact
score's visibility, intensity and wind components rather than by cohort membership.

In [ ]:
CHOSEN_SCHEME = "baseline"
K = 9

Z = Z_base * scheme_weights(CHOSEN_SCHEME)
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10).fit(Z)
labels = kmeans.labels_
assert adjusted_rand_score(labels, grid_cache["labels"][(CHOSEN_SCHEME, K)]) == 1.0
profile = cohort_profile(labels)

side_by_side = grid.set_index(["scheme", "k"]).loc[[("baseline", 9), (CHOSEN_SCHEME, K)]]
display(side_by_side[["silhouette", "ari_min_vs_seeds", "unnamed", "duplicate_names", "heavy_precip_own",
                      "dense_fog_own", "high_wind_own", "labels"]])

# ambiguity: nearest / second-nearest centroid distance (threshold var weather_cohort_ambiguity_threshold)
AMBIGUITY_THRESHOLD = 0.8
dist = np.sort(kmeans.transform(Z), axis=1)
distance_ratio = dist[:, 0] / dist[:, 1]
ambiguous = distance_ratio > AMBIGUITY_THRESHOLD
w = sample["sample_weight"].to_numpy()
print(f"distance ratio p50/p90/p95/p99: {np.round(np.quantile(distance_ratio, [0.5, 0.9, 0.95, 0.99]), 3)}")
print(f"ambiguous (ratio > {AMBIGUITY_THRESHOLD}): {ambiguous.mean():.2%} of sample rows, "
      f"{np.average(ambiguous, weights=w):.2%} of population")
print("ambiguous share by cohort (population-weighted):")
(pd.DataFrame({"cohort_label": profile["cohort_label"].reindex(labels).values, "ambiguous": ambiguous, "w": w})
   .groupby("cohort_label").apply(lambda g: np.average(g["ambiguous"], weights=g["w"]), include_groups=False)
   .sort_values(ascending=False).round(4))

## 5. Gaussian Mixture comparison

A GMM with the same k on the same (weighted) features, fit independently (k-means++ init). Agreement
with K-Means is measured with ARI / NMI. GMM is **not** used for `ambiguous_cohort` (its top component
probability is ~1.0 for almost every row here, so it can't separate ambiguous rows); the K-Means distance
ratio above is used instead.

In [ ]:
from sklearn.mixture import GaussianMixture

t0 = time.time()
gmm = GaussianMixture(n_components=K, covariance_type="full", random_state=RANDOM_STATE,
                      reg_covar=1e-3, max_iter=200, init_params="k-means++").fit(Z)
g_labels = gmm.predict(Z)
top_prob = gmm.predict_proba(Z).max(axis=1)
print(f"GMM full: {time.time() - t0:.0f}s converged={gmm.converged_} "
      f"ARI={adjusted_rand_score(labels, g_labels):.3f} NMI={normalized_mutual_info_score(labels, g_labels):.3f}")
print(f"GMM top-prob p1/p5/p50: {np.round(np.quantile(top_prob, [0.01, 0.05, 0.5]), 3)}")
ct = pd.crosstab(pd.Series(profile["cohort_label"].reindex(labels).values, name="K-Means cohort"),
                 pd.Series(g_labels, name="GMM component"))
ct_share = ct.div(ct.sum(axis=1), axis=0)
plt.figure(figsize=(10, 6))
sns.heatmap(ct_share, annot=True, fmt=".2f", cmap="Blues", cbar=False)
plt.title("Share of each K-Means cohort in each GMM component")
plt.show()

## 6. HDBSCAN: rare extreme-weather outliers

HDBSCAN on a 100k-row random subset of the sample, in the all-standardized space (no k). Points labeled
-1 are noise: rows that don't sit in any dense region. The question is whether they are meaningful
extreme weather, and whether the chosen cohorts capture them.

In [ ]:
from sklearn.cluster import HDBSCAN

HDBSCAN_ROWS = 100_000
hdb_idx = np.random.default_rng(RANDOM_STATE).choice(len(Z_base), HDBSCAN_ROWS, replace=False)
t0 = time.time()
hdb_labels = HDBSCAN(min_cluster_size=500, min_samples=25).fit(Z_base[hdb_idx]).labels_
is_noise = hdb_labels == -1
print(f"HDBSCAN on {HDBSCAN_ROWS:,} rows: {time.time() - t0:.0f}s, "
      f"{hdb_labels.max() + 1} clusters, noise share {is_noise.mean():.1%}")

raw = sample.iloc[hdb_idx]
tail = pd.DataFrame({
    "heavy intensity": raw["precip_intensity"].eq("heavy").to_numpy(),
    "visibility < 1 mi": (raw["visibility_miles_clipped"] < 1).to_numpy(),
    "wind >= 25 mph": (raw["wind_speed_mph_clipped"] >= 25).to_numpy(),
    "precip >= 0.3 in": (raw["precipitation_inches_capped"] >= 0.3).to_numpy(),
    "blowing": raw["is_blowing"].to_numpy(),
})
display(pd.DataFrame({"noise": tail[is_noise].mean(), "clustered": tail[~is_noise].mean()}).round(3))
print("noise share by precip_type:")
display(pd.Series(is_noise).groupby(raw["precip_type"].to_numpy()).mean().sort_values(ascending=False).round(3))
print("where the HDBSCAN noise rows land in the chosen cohorts:")
pd.Series(profile["cohort_label"].reindex(labels[hdb_idx]).values[is_noise]).value_counts()

## 7. Cohort profiles and impact score calibration

Profiles are **weighted** by `sample_weight`, so they describe the population rather than the
oversampled sample. Labels come from the rules in 4b on each cluster's average profile, not from cluster
ids, so they stay stable across refits; the same rules will be implemented in dbt.

In [ ]:
heat_cols = NUMERIC_FEATURES + ["precip_intensity_ord"] + FLAG_FEATURES + list(markers.columns) + TYPE_COLUMNS
heat = profile.set_index("cohort_label")[heat_cols]
plt.figure(figsize=(18, 0.6 * K + 2))
sns.heatmap((heat - heat.mean()) / heat.std(ddof=0), annot=heat.round(2), fmt="", cmap="RdBu_r", center=0, cbar=False)
plt.title("Weighted cohort profiles (color: relative to other cohorts; numbers: mean values / shares)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()
profile.sort_values("population_share", ascending=False)[["cohort_label", "population_share", "sample_rows"]].round(4)

### Impact score: worst hazard wins

Three hazards, each 0-10, from lookup tables (to be stored as a dbt seed / vars so they're easy to adjust):

- **visibility hazard**: piecewise-linear in `visibility_miles_clipped` (interpolated between points)
- **precipitation sensor hazard**: `precip_type` x `precip_intensity` (rain and snow block cameras and lidar
  even when visibility is reported as good); fog, haze and clear are 0 because visibility covers them
- **traction hazard**: first matching rule of freezing precipitation (`is_freezing`, precipitating and
  at or below 34F; above 34F it gets the black ice value) ->
  snow by intensity -> `freezing_wet` (black ice risk) -> heavy rain (hydroplaning) -> wet road -> dry

`base = max(hazards) + 0.25 x second-highest` (so fog plus ice scores above either alone), then darkness
+1.0, wind +0 below 25 mph rising linearly to +1.5 at 50 mph, blowing +1.0; clipped to 0-10, one decimal.

The lookup values started from the proposed points and were tuned (constrained search plus hand
adjustment) to put every anchor within +-0.5 while keeping each table monotonic, rain <= snow at each
intensity, and snow traction >= freezing_wet traction. The checks below assert those properties.

In [ ]:
# --- lookup tables (proposed seed: weather_impact_hazard_lookup.csv)
VIS_POINTS = [(0.1, 9.0), (0.25, 7.5), (0.5, 5.5), (1.0, 3.5), (3.0, 1.5), (5.0, 0.0)]   # (miles, hazard)
PRECIP_SENSOR = {                                                                          # (type, intensity) -> hazard
    ("rain", "light"): 2.25, ("rain", "moderate"): 3.5, ("rain", "heavy"): 4.75,
    ("snow", "light"): 2.25, ("snow", "moderate"): 6.0, ("snow", "heavy"): 7.5,
    ("ice_freezing", "light"): 5.0, ("ice_freezing", "moderate"): 5.0, ("ice_freezing", "heavy"): 5.0,
    ("thunderstorm", "light"): 3.5, ("thunderstorm", "moderate"): 3.5, ("thunderstorm", "heavy"): 4.75,
}
FREEZING_THRESHOLD_F = 34   # var weather_freezing_threshold_f
TRACTION = {"freezing_precip": 8.0, "snow_light": 4.75, "snow_moderate": 5.75, "snow_heavy": 7.5,
            "freezing_wet": 4.75, "heavy_rain": 3.0, "wet_road": 2.0, "dry": 0.0}
SECOND_HAZARD_BONUS = 0.25
DARKNESS_ADD = 1.0
WIND_ADD_START_MPH, WIND_ADD_FULL_MPH, WIND_ADD_MAX = 25.0, 50.0, 1.5
BLOWING_ADD = 1.0
LEVEL_CUTOFFS = [(8.0, "Severe"), (6.0, "High"), (3.0, "Moderate"), (0.0, "Low")]
HAZARDS = ["visibility_hazard", "precip_sensor_hazard", "traction_hazard"]

# monotonic checks
vis_y = [h for _, h in VIS_POINTS]
assert all(a >= b for a, b in zip(vis_y, vis_y[1:])), "visibility hazard must fall as visibility rises"
for t in ["rain", "snow", "ice_freezing", "thunderstorm"]:
    seq = [PRECIP_SENSOR[(t, i)] for i in ["light", "moderate", "heavy"]]
    assert seq == sorted(seq), f"{t} sensor hazard must rise with intensity"
assert all(PRECIP_SENSOR[("snow", i)] >= PRECIP_SENSOR[("rain", i)] for i in ["light", "moderate", "heavy"])
chain = [TRACTION[k] for k in ["dry", "wet_road", "heavy_rain", "freezing_wet", "snow_light", "snow_moderate", "snow_heavy", "freezing_precip"]]
assert chain == sorted(chain), "traction table must be monotonic"


def impact_components(d: pd.DataFrame) -> pd.DataFrame:
    vis = np.interp(d["visibility_miles_clipped"].astype(float), *zip(*VIS_POINTS))
    keys = list(zip(d["precip_type"], d["precip_intensity"]))
    sensor = np.array([PRECIP_SENSOR.get(k, 0.0) for k in keys])
    ptype, inten = d["precip_type"], d["precip_intensity"]
    precipitating = d["is_precipitating"].fillna(False).astype(bool) if "is_precipitating" in d else pd.Series(False, index=d.index)
    freezing_precip = d["is_freezing"].astype(bool) & precipitating
    traction = np.select(
        [freezing_precip & (d["temperature_fahrenheit_clipped"].astype(float) <= FREEZING_THRESHOLD_F),
         ptype.eq("snow") & inten.eq("heavy"), ptype.eq("snow") & inten.eq("moderate"), ptype.eq("snow") & inten.eq("light"),
         d["freezing_wet"].astype(bool) | freezing_precip,   # freezing precip above the threshold -> black ice value
         ptype.isin(["rain", "thunderstorm"]) & inten.eq("heavy"),
         d["wet_road"].astype(bool)],
        [TRACTION["freezing_precip"], TRACTION["snow_heavy"], TRACTION["snow_moderate"], TRACTION["snow_light"],
         TRACTION["freezing_wet"], TRACTION["heavy_rain"], TRACTION["wet_road"]],
        TRACTION["dry"])
    out = pd.DataFrame({"visibility_hazard": vis, "precip_sensor_hazard": sensor, "traction_hazard": traction}, index=d.index)
    h = np.sort(out[HAZARDS].to_numpy(), axis=1)
    names = np.array([name.replace("_hazard", "") for name in HAZARDS])
    out["driver"] = np.where(h[:, -1] > 0, names[out[HAZARDS].to_numpy().argmax(axis=1)], "none")
    out["base"] = h[:, -1] + SECOND_HAZARD_BONUS * h[:, -2]
    out["darkness_add"] = d["is_night"].astype(float).to_numpy() * DARKNESS_ADD
    wind = d["wind_speed_mph_clipped"].astype(float).to_numpy()
    out["wind_add"] = WIND_ADD_MAX * np.clip((wind - WIND_ADD_START_MPH) / (WIND_ADD_FULL_MPH - WIND_ADD_START_MPH), 0, 1)
    out["blowing_add"] = d["is_blowing"].astype(float).to_numpy() * BLOWING_ADD
    # round half up to one decimal, like BigQuery ROUND (pandas .round() rounds half to even)
    raw = (out["base"] + out["darkness_add"] + out["wind_add"] + out["blowing_add"]).clip(0, 10)
    out["score"] = np.floor(raw * 10 + 0.5) / 10
    out["level"] = np.select([out["score"] >= c for c, _ in LEVEL_CUTOFFS], [l for _, l in LEVEL_CUTOFFS], "Low")
    return out


def impact_score(d):
    return impact_components(d)

In [ ]:
def typical(mask, **fixed):
    """Typical row for a scenario: medians of numeric columns, majority of flags, mode of categories."""
    rows = sample[mask]
    out = {
        "visibility_miles_clipped": rows["visibility_miles_clipped"].median(),
        "wind_speed_mph_clipped": rows["wind_speed_mph_clipped"].median(),
        "temperature_fahrenheit_clipped": rows["temperature_fahrenheit_clipped"].median(),
        "precip_type": rows["precip_type"].mode().iloc[0],
        "precip_intensity": rows["precip_intensity"].mode().iloc[0],
        "is_precipitating": bool(rows["is_precipitating"].fillna(False).mean() >= 0.5),
    }
    for f in ["wet_road", "freezing_wet", "is_blowing", "is_night", "is_freezing"]:
        out[f] = bool(rows[f].mean() >= 0.5)
    out.update(fixed)
    out["matching_sample_rows"] = int(mask.sum())
    return out


s = sample
day, night = ~s["is_night"], s["is_night"]
clear_dry = (s.precip_type == "clear_cloudy") & ~s.wet_road
ANCHORS = [
    ("Clear, dry, daytime", 0, typical(clear_dry & day, is_night=False)),
    ("Clear, dry, night", 1, typical(clear_dry & night, is_night=True)),
    ("Light rain, daytime", 3, typical((s.precip_type == "rain") & (s.precip_intensity == "light") & day, is_night=False)),
    ("Heavy rain, daytime, 5 mi", 5, typical((s.precip_type == "rain") & (s.precip_intensity == "heavy") & day,
                                              visibility_miles_clipped=5.0, is_night=False)),
    ("Heavy rain, night", 7, typical((s.precip_type == "rain") & (s.precip_intensity == "heavy") & night, is_night=True)),
    ("Black ice (clear, wet, <= 34F, night)", 6, typical(clear_dry & night, visibility_miles_clipped=10.0, wet_road=True,
                                                         temperature_fahrenheit_clipped=30.0,
                                                         freezing_wet=True, is_freezing=False, is_night=True)),
    ("Light snow, 5 mi", 5, typical((s.precip_type == "snow") & (s.precip_intensity == "light"),
                                    visibility_miles_clipped=5.0, is_night=False)),
    ("Moderate snow", 7, typical((s.precip_type == "snow") & (s.precip_intensity == "moderate"), is_night=False)),
    ("Dense fog (0.2 mi)", 8, typical((s.precip_type == "fog") & (s.visibility_miles_clipped < 0.25),
                                      visibility_miles_clipped=0.2, is_night=False)),
    ("Heavy snow", 9, typical((s.precip_type == "snow") & (s.precip_intensity == "heavy"), is_night=False)),
    ("Freezing rain", 9, typical(s.weather_condition.fillna("").str.contains("freezing rain"), is_night=False)),
]
anchors = pd.DataFrame([{"scenario": n, "target": t, **v} for n, t, v in ANCHORS]).set_index("scenario")
anchor_scores = impact_components(anchors)
anchor_table = anchors[["matching_sample_rows", "visibility_miles_clipped", "wind_speed_mph_clipped", "temperature_fahrenheit_clipped", "precip_type",
                        "precip_intensity", "wet_road", "freezing_wet", "is_freezing", "is_night", "target"]].join(
    anchor_scores[HAZARDS + ["driver", "base", "darkness_add", "wind_add", "score", "level"]])
anchor_table["error"] = anchor_table["score"] - anchor_table["target"]
assert anchor_table["error"].abs().max() <= 0.5 + 1e-9, "an anchor is outside +-0.5"
anchor_table.round(2)

In [ ]:
print("visibility hazard points (miles -> hazard):", VIS_POINTS)
print("precipitation sensor hazard:")
display(pd.Series(PRECIP_SENSOR).unstack().reindex(columns=["light", "moderate", "heavy"]))
print("traction hazard:", TRACTION)
print(f"base = max + {SECOND_HAZARD_BONUS} x second; darkness +{DARKNESS_ADD}; wind +0 below {WIND_ADD_START_MPH:.0f} mph "
      f"to +{WIND_ADD_MAX} at {WIND_ADD_FULL_MPH:.0f} mph; blowing +{BLOWING_ADD}")

In [ ]:
scores = impact_score(sample)
scored = sample.assign(score=scores["score"].to_numpy(), level=scores["level"],
                       cohort_label=profile["cohort_label"].reindex(labels).values)
w = scored["sample_weight"]

fig, ax = plt.subplots(1, 2, figsize=(15, 0.35 * K + 3))
ax[0].hist(scored["score"], bins=np.arange(0, 10.2, 0.2), weights=w)
ax[0].set_yscale("log"); ax[0].set_title("Impact score, population-weighted (log scale)"); ax[0].set_xlabel("score")
cohort_mean = scored.groupby("cohort_label").apply(lambda g: np.average(g.score, weights=g.sample_weight), include_groups=False).sort_values()
sns.boxplot(data=scored, y="cohort_label", x="score", order=cohort_mean.index, ax=ax[1], showfliers=False)
ax[1].set_title("Impact score by cohort (sample)")
plt.tight_layout(); plt.show()

levels = ["Low", "Moderate", "High", "Severe"]
level_share = (w.groupby(scored["level"]).sum() / w.sum()).reindex(levels).fillna(0)
print("population share by level (known precip_type rows):", level_share.round(4).to_dict())
by_cohort = (scored.assign(w=w).pivot_table(index="cohort_label", columns="level", values="w", aggfunc="sum", fill_value=0)
             .reindex(columns=levels, fill_value=0))
by_cohort = by_cohort.div(by_cohort.sum(axis=1), axis=0).round(3)
by_cohort.insert(0, "mean_score", cohort_mean.round(2))
by_cohort.sort_values("mean_score")

### Spot check: 10 random rows per level

Sampled uniformly within each level from the stratified sample (`random_state = 42`).

In [ ]:
spot_cols = ["construction_id", "cohort_label", "precip_type", "precip_intensity", "weather_condition",
             "visibility_miles_clipped", "wind_speed_mph_clipped", "temperature_fahrenheit_clipped",
             "wet_road", "freezing_wet", "is_freezing", "is_night", "is_blowing"]
comp_cols = HAZARDS + ["driver", "base", "darkness_add", "wind_add", "blowing_add", "score"]
spot = scored[spot_cols].join(scores[comp_cols].drop(columns="score")).assign(score=scored["score"], level=scored["level"])
for lvl in ["Low", "Moderate", "High", "Severe"]:
    print(f"--- {lvl}")
    pool = spot[spot["level"] == lvl]
    display(pool.sample(min(10, len(pool)), random_state=RANDOM_STATE).drop(columns="level").round(2))

## 8. Conclusions

Values to copy into dbt vars (`dbt_project.yml`). Centroids are reported in original units (inverse of
the weighting and standardization) so the label rules can be checked against them.

In [ ]:
col_w = scheme_weights(CHOSEN_SCHEME)
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_ / col_w), columns=X.columns)
centroids.insert(0, "cohort_label", profile["cohort_label"].values)

print(f"weather_cohort_k: {K}")
print(f"weather_cohort_random_state: {RANDOM_STATE}")
print(f"weather_cohort_feature_weights ({CHOSEN_SCHEME}):", {c: round(v, 4) for c, v in WEIGHT_SCHEMES[CHOSEN_SCHEME].items()} or "all 1.0")
print(f"weather_cohort_ambiguity_threshold: {AMBIGUITY_THRESHOLD}")
print("weather_impact_visibility_points:", VIS_POINTS)
print("weather_impact_precip_sensor:", {f"{k[0]}/{k[1]}": v for k, v in PRECIP_SENSOR.items()})
print("weather_impact_traction:", TRACTION)
print("weather_impact_additions:", {"second_hazard_bonus": SECOND_HAZARD_BONUS, "darkness": DARKNESS_ADD,
      "wind_start_mph": WIND_ADD_START_MPH, "wind_full_mph": WIND_ADD_FULL_MPH, "wind_max": WIND_ADD_MAX, "blowing": BLOWING_ADD})
print("weather_impact_level_cutoffs:", {l: c for c, l in LEVEL_CUTOFFS})
print("proposed cohort labels (by population share):")
for _, r in profile.sort_values("population_share", ascending=False).iterrows():
    print(f"  - {r.cohort_label:24s} {r.population_share:7.2%} of population")
centroids.round(2)

### Hand-off to BigQuery ML

BigQuery ML k-means can start from given centroids (`kmeans_init_method = 'CUSTOM'`), but they must be rows
of the training data. For each notebook cluster, the training row nearest its centroid is written to the
dbt seed `seeds/weather_cohort_init_centroids.csv`; the notebook's cluster per sample row is saved to
`notebooks/data/notebook_labels.parquet` to measure agreement with BigQuery ML.

In [ ]:
nearest_rows = np.argmin(kmeans.transform(Z), axis=0)          # one training row per cluster
init = pd.DataFrame({
    "construction_id": sample["construction_id"].iloc[nearest_rows].to_numpy(),
    "notebook_cluster": np.arange(K),
    "cohort_label": profile["cohort_label"].reindex(np.arange(K)).to_numpy(),
})
assert (labels[nearest_rows] == np.arange(K)).all(), "each init row must belong to its own cluster"
SEED_PATH = NOTEBOOK_DIR.parent / "road_construction_autonomy" / "seeds" / "weather_cohort_init_centroids.csv"
init.sort_values("notebook_cluster").to_csv(SEED_PATH, index=False)
pd.DataFrame({"construction_id": sample["construction_id"].to_numpy(), "notebook_cluster": labels,
              "cohort_label": profile["cohort_label"].reindex(labels).to_numpy()}).to_parquet(DATA_DIR / "notebook_labels.parquet", index=False)
print(f"wrote {SEED_PATH}")
init